# A2-Lite — two smaller-input speed screens
Latest revision: **A2-LITE-RESOLUTION-ONNX-CPU-SCREEN-2026-10-12-r1**

Keep the exact reviewed E1/D1 pruned A2 weights, MobileNetV4 Medium backbone, native depth module,256 channels,50 queries and FP32 ONNX CPU execution. Change only image resolution: **960×288** and **640×192** (width×height), versus1280×384. No training, distillation, CUDA extension build, Core ML conversion, old runtime restore or KITTI download. CPU runtime is sufficient.

This notebook uses the original A2-Lite Drive folder, including a2_lite_initialization.pth (excluded from the earlier download ZIP). It resizes the reviewed normalized tensors without cropping, keeps original calibration/image_size inputs unchanged, updates the portable attention feature grids, and verifies raw AND decoded candidate ONNX/native parity on all16 fixtures. Smaller-input outputs are not expected to equal full-resolution outputs. These fixtures are for speed screening, NOT an accuracy evaluation.

Run sections1–5 in order. Preserve previous outputs; choose a fresh RUN_ID after a partial export or code change. Return a2_lite_resolution_results.zip. After review, each candidate will be added as a separate option in the existing phone app. Do not replace A2_ONNX or A2_LITE_ONNX. Phone targets remain model p95≤50ms and sustained processed FPS≥10; host timing cannot pass those gates. Smaller images may reduce pedestrian accuracy; accuracy/recovery training comes only after the speed screen.

## 1. Mount Drive, set paths and define every helper
BASELINE is the folder from the completed full-resolution A2-Lite notebook, not the downloaded ZIP. No cells from another notebook are required.

In [ ]:
# A2-LITE-RESOLUTION-ONNX-CPU-SCREEN-2026-10-12-r1
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
from datetime import datetime, timezone
from collections import deque
import json, re, shlex, subprocess, sys
REVISION='A2-LITE-RESOLUTION-ONNX-CPU-SCREEN-2026-10-12-r1'
RUN_ID='a2_lite_resolution_cpu_screen_r1'
if not re.fullmatch(r'[a-z0-9_]+',RUN_ID): raise ValueError('Use lowercase letters, digits and underscores in RUN_ID')
MOBILE_REPO=Path('/content/mobile_adas3d')
BASELINE=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite/a2_lite_e1_d1_cpu_screen_r1')
OUTPUT=Path('/content/drive/MyDrive/mobile_adas3d_outputs/edge_candidates/a2_lite_resolution')/RUN_ID
SOURCE_REPO=Path('/content/MonoDETR_A2LITE_'+RUN_ID)
PYTHON=Path(sys.executable)
SCRIPT=MOBILE_REPO/'scripts/a2_lite_resolution_onnx.py'
LOGS=OUTPUT/'colab_logs'; LOGS.mkdir(parents=True,exist_ok=True)
READY=False
def run_logged(command,name,cwd=None):
    command=list(map(str,command))
    path=LOGS/f"{name}_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S_%fZ')}.log"
    print('+',shlex.join(command),'\nDurable log:',path,flush=True)
    tail=deque(maxlen=40)
    with path.open('w') as log:
        log.write(shlex.join(command)+'\n'); log.flush()
        process=subprocess.Popen(command,cwd=str(cwd) if cwd else None,stdout=subprocess.PIPE,stderr=subprocess.STDOUT,text=True,bufsize=1)
        try:
            for line in process.stdout:
                print(line,end='',flush=True); log.write(line); log.flush(); tail.append(line.rstrip())
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
        finally: process.stdout.close()
        code=process.wait(); log.write(f'\nEXIT CODE: {code}\n'); log.flush()
    if code: raise RuntimeError(f'Exit {code}; full log={path}\n'+'\n'.join(tail))
    return path
def stage(name,*extra):
    if not READY: raise RuntimeError('Run sections1 and2 first')
    return run_logged([PYTHON,'-u',SCRIPT,name,'--output',OUTPUT,*extra],name,MOBILE_REPO)
print(REVISION,'\nBaseline:',BASELINE,'\nNew output:',OUTPUT)


## 2. Update clean main and install CPU export dependencies
Torch/CUDA are not installed or replaced. The subprocesses use the current Python with ONNX Runtime1.30.0; no historical GPU identity is required.

In [ ]:
READY=False
if not MOBILE_REPO.exists(): run_logged(['git','clone','--branch','main','https://github.com/Ali-RT/mobile_adas3d.git',MOBILE_REPO],'clone')
dirty=subprocess.check_output(['git','status','--porcelain','--untracked-files=no'],cwd=MOBILE_REPO,text=True).strip()
if dirty: raise RuntimeError('Preserve tracked edits; use a clean main checkout.\n'+dirty)
branch=subprocess.check_output(['git','branch','--show-current'],cwd=MOBILE_REPO,text=True).strip()
if branch!='main': raise RuntimeError(f'Use main, not {branch!r}')
run_logged(['git','pull','--ff-only','origin','main'],'update',MOBILE_REPO)
if not SCRIPT.is_file(): raise FileNotFoundError(SCRIPT)
for path in (BASELINE/'a2_lite_manifest.json',BASELINE/'a2_lite_initialization.pth',BASELINE/'export/A2_LITE_ONNX/manifest.json'):
    if not path.is_file(): raise FileNotFoundError(f'{path}: use the original Drive output folder, not the download ZIP')
run_logged([PYTHON,'-m','pip','install','-q','timm==1.0.27','numpy==2.1.3','onnx==1.20.1','onnxruntime==1.30.0','pyyaml','scipy','pillow','opencv-python-headless'],'dependencies')
run_logged([PYTHON,'-c',"import torch,timm,onnx,onnxruntime as ort; assert timm.__version__=='1.0.27'; assert ort.__version__=='1.30.0'; print(torch.__version__,timm.__version__,onnx.__version__,ort.__version__)"],'versions')
READY=True


## 3. Tests and exact baseline verification
Verify the original signed A2-Lite metadata, checkpoint SHA256 and all16 baseline CPU fixtures. Restore only a dedicated portable source checkout; never the old virtual environment.

In [ ]:
run_logged([PYTHON,'-m','unittest','discover','-s','tests','-p','test_a2_lite_resolution.py','-v'],'tests',MOBILE_REPO)
stage('prepare','--baseline',BASELINE,'--repo',SOURCE_REPO)


## 4. Export both candidates and validate each independently
Each subprocess strictly loads the SAME pruned checkpoint, verifies the reconstructed baseline, changes image/attention-grid dimensions, and checks that all model tensors remain unchanged. Candidate-native and ONNX outputs must match at the NEW resolution, including decoded metric geometry. Host CPU timing is exploratory; no accuracy or iPhone qualification is claimed.

In [ ]:
for variant in ('r960x288','r640x192'):
    stage('export','--variant',variant)
    report=json.loads((OUTPUT/'exports'/variant/'a2_lite_resolution_export.json').read_text())
    assert report['complete'] and report['model_weights_unchanged'] and report['feature_grids_verified']
    assert len(report['checks'])==16 and not report['training_performed'] and report['optimizer_steps']==0
    print(json.dumps({k:report[k] for k in ('variant_id','feature_spatial_shapes','model_weights_unchanged','host_cpu_timing')},indent=2))
print('Both exports ready for review. Actual iPhone timing comes after the returned artifacts are checked.')


## 5. Download the resolution-screen bundle
Also run this section after a failure to return its full logs. The bundle excludes checkpoints and original baseline assets. Return it for review and separate phone integration; no new phone test is ready until the exact candidate hashes are pinned.

In [ ]:
stage('bundle')
BUNDLE=OUTPUT/'a2_lite_resolution_results.zip'
print(BUNDLE)
from google.colab import files
files.download(str(BUNDLE))
